# GraphCast

This colab lets you run several versions of GraphCast.

The model weights, normalization statistics, and example inputs are available on [Google Cloud Bucket](https://console.cloud.google.com/storage/browser/dm_graphcast).

A Colab runtime with TPU/GPU acceleration will substantially speed up generating predictions and computing the loss/gradients. If you're using a CPU-only runtime, you can switch using the menu "Runtime > Change runtime type".

> <p><small><small>Copyright 2023 DeepMind Technologies Limited.</small></p>
> <p><small><small>Licensed under the Apache License, Version 2.0 (the "License"); you may not use this file except in compliance with the License. You may obtain a copy of the License at <a href="http://www.apache.org/licenses/LICENSE-2.0">http://www.apache.org/licenses/LICENSE-2.0</a>.</small></small></p>
> <p><small><small>Unless required by applicable law or agreed to in writing, software distributed under the License is distributed on an "AS IS" BASIS, WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied. See the License for the specific language governing permissions and limitations under the License.</small></small></p>

# Installation and Initialization


In [ ]:
# @title Pip install graphcast and dependencies
%pip install --upgrade https://github.com/deepmind/graphcast/archive/master.zip

In [ ]:
%pip install netCDF4

In [ ]:
# @title Imports

import dataclasses
import datetime
import functools
import math
import re
from typing import Optional
import cartopy.crs as ccrs
from google.cloud import storage
from google.colab import auth
from graphcast import autoregressive
from graphcast import casting
from graphcast import checkpoint
from graphcast import data_utils
from graphcast import graphcast
from graphcast import normalization
from graphcast import rollout
from graphcast import xarray_jax
from graphcast import xarray_tree
from IPython.display import HTML
import ipywidgets as widgets
import haiku as hk
import jax
import matplotlib
import matplotlib.pyplot as plt
from matplotlib import animation
import numpy as np
import xarray
#from perlin_numpy import generate_fractal_noise_2d, generate_fractal_noise_3d,generate_perlin_noise_2d, generate_perlin_noise_3d

def parse_file_parts(file_name):
  return dict(part.split("-", 1) for part in file_name.split("_"))

In [ ]:
# @title Authenticate with Google Cloud Storage

gcs_client = storage.Client.create_anonymous_client()
gcs_bucket = gcs_client.get_bucket("dm_graphcast")

In [ ]:
# @title Plotting functions

def select(
    data: xarray.Dataset,
    variable: str,
    level: Optional[int] = None,
    max_steps: Optional[int] = None
    ) -> xarray.Dataset:
  data = data[variable]
  if "batch" in data.dims:
    data = data.isel(batch=0)
  if max_steps is not None and "time" in data.sizes and max_steps < data.sizes["time"]:
    data = data.isel(time=range(0, max_steps))
  if level is not None and "level" in data.coords:
    data = data.sel(level=level)
  return data

def scale(
    data: xarray.Dataset,
    center: Optional[float] = None,
    robust: bool = False,
    ) -> tuple[xarray.Dataset, matplotlib.colors.Normalize, str]:
  vmin = np.nanpercentile(data, (2 if robust else 0))
  vmax = np.nanpercentile(data, (98 if robust else 100))
  if center is not None:
    diff = max(vmax - center, center - vmin)
    vmin = center - diff
    vmax = center + diff
  return (data, matplotlib.colors.Normalize(vmin, vmax),
          ("RdBu_r" if center is not None else "viridis"))

def plot_data(
    data: dict[str, xarray.Dataset],
    fig_title: str,
    plot_size: float = 5,
    robust: bool = False,
    cols: int = 4
    ) -> tuple[xarray.Dataset, matplotlib.colors.Normalize, str]:

  first_data = next(iter(data.values()))[0]
  max_steps = first_data.sizes.get("time", 1)
  assert all(max_steps == d.sizes.get("time", 1) for d, _, _ in data.values())

  cols = min(cols, len(data))
  rows = math.ceil(len(data) / cols)
  figure = plt.figure(figsize=(plot_size * 2 * cols,
                               plot_size * rows))
  figure.suptitle(fig_title, fontsize=16)
  figure.subplots_adjust(wspace=0, hspace=0)
  figure.tight_layout()

  images = []
  for i, (title, (plot_data, norm, cmap)) in enumerate(data.items()):
    ax = figure.add_subplot(rows, cols, i+1)
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_title(title)
    im = ax.imshow(
        plot_data.isel(time=0, missing_dims="ignore"), norm=norm,
        origin="lower", cmap=cmap)
    plt.colorbar(
        mappable=im,
        ax=ax,
        orientation="vertical",
        pad=0.02,
        aspect=16,
        shrink=0.75,
        cmap=cmap,
        extend=("both" if robust else "neither"))
    images.append(im)

  def update(frame):
    if "time" in first_data.dims:
      td = datetime.timedelta(microseconds=first_data["time"][frame].item() / 1000)
      figure.suptitle(f"{fig_title}, {td}", fontsize=16)
    else:
      figure.suptitle(fig_title, fontsize=16)
    for im, (plot_data, norm, cmap) in zip(images, data.values()):
      im.set_data(plot_data.isel(time=frame, missing_dims="ignore"))

  ani = animation.FuncAnimation(
      fig=figure, func=update, frames=max_steps, interval=250)
  plt.close(figure.number)
  return HTML(ani.to_jshtml())

# Load the Data and initialize the model

## Load the model params

Choose one of the two ways of getting model params:
- **random**: You'll get random predictions, but you can change the model architecture, which may run faster or fit on your device.
- **checkpoint**: You'll get sensible predictions, but are limited to the model architecture that it was trained with, which may not fit on your device. In particular generating gradients uses a lot of memory, so you'll need at least 25GB of ram (TPUv4 or A100).

Checkpoints vary across a few axes:
- The mesh size specifies the internal graph representation of the earth. Smaller meshes will run faster but will have worse outputs. The mesh size does not affect the number of parameters of the model.
- The resolution and number of pressure levels must match the data. Lower resolution and fewer levels will run a bit faster. Data resolution only affects the encoder/decoder.
- All our models predict precipitation. However, ERA5 includes precipitation, while HRES does not. Our models marked as "ERA5" take precipitation as input and expect ERA5 data as input, while model marked "ERA5-HRES" do not take precipitation as input and are specifically trained to take HRES-fc0 as input (see the data section below).

We provide three pre-trained models.
1. `GraphCast`, the high-resolution model used in the GraphCast paper (0.25 degree resolution, 37 pressure levels), trained on ERA5 data from 1979 to 2017,

2. `GraphCast_small`, a smaller, low-resolution version of GraphCast (1 degree resolution, 13 pressure levels, and a smaller mesh), trained on ERA5 data from 1979 to 2015, useful to run a model with lower memory and compute constraints,

3. `GraphCast_operational`, a high-resolution model (0.25 degree resolution, 13 pressure levels) pre-trained on ERA5 data from 1979 to 2017 and fine-tuned on HRES data from 2016 to 2021. This model can be initialized from HRES data (does not require precipitation inputs).


In [ ]:
# @title Choose the model

params_file_options = [
    name for blob in gcs_bucket.list_blobs(prefix="params/")
    if (name := blob.name.removeprefix("params/"))]  # Drop empty string.

random_mesh_size = widgets.IntSlider(
    value=4, min=4, max=6, description="Mesh size:")
random_gnn_msg_steps = widgets.IntSlider(
    value=4, min=1, max=32, description="GNN message steps:")
random_latent_size = widgets.Dropdown(
    options=[int(2**i) for i in range(4, 10)], value=32,description="Latent size:")
random_levels = widgets.Dropdown(
    options=[13, 37], value=13, description="Pressure levels:")

params_file = widgets.Dropdown(
    options=params_file_options,
    description="Params file:",
    layout={"width": "max-content"})

source_tab = widgets.Tab([
    widgets.VBox([
        random_mesh_size,
        random_gnn_msg_steps,
        random_latent_size,
        random_levels,
    ]),
    params_file,
])
source_tab.set_title(0, "Random")
source_tab.set_title(1, "Checkpoint")
widgets.VBox([
    source_tab,
    widgets.Label(value="Run the next cell to load the model. Rerunning this cell clears your selection.")
])


In [ ]:
# @title Load the model

source = source_tab.get_title(source_tab.selected_index)

if source == "Random":
  params = params_file  # Filled in below
  state = {}
  model_config = graphcast.ModelConfig(
      resolution=0,
      mesh_size=random_mesh_size.value,
      latent_size=random_latent_size.value,
      gnn_msg_steps=random_gnn_msg_steps.value,
      hidden_layers=1,
      radius_query_fraction_edge_length=0.6)
  task_config = graphcast.TaskConfig(
      input_variables=graphcast.TASK.input_variables,
      target_variables=graphcast.TASK.target_variables,
      forcing_variables=graphcast.TASK.forcing_variables,
      pressure_levels=graphcast.PRESSURE_LEVELS[random_levels.value],
      input_duration=graphcast.TASK.input_duration,
  )
else:
  assert source == "Checkpoint"
  with gcs_bucket.blob(f"params/{params_file.value}").open("rb") as f:
    ckpt = checkpoint.load(f, graphcast.CheckPoint)
  params = ckpt.params
  state = {}

  model_config = ckpt.model_config
  task_config = ckpt.task_config
  print("Model description:\n", ckpt.description, "\n")
  print("Model license:\n", ckpt.license, "\n")

model_config

## Load the example data

Several example datasets are available, varying across a few axes:
- **Source**: fake, era5, hres
- **Resolution**: 0.25deg, 1deg, 6deg
- **Levels**: 13, 37
- **Steps**: How many timesteps are included

Not all combinations are available.
- Higher resolution is only available for fewer steps due to the memory requirements of loading them.
- HRES is only available in 0.25 deg, with 13 pressure levels.

The data resolution must match the model that is loaded.

Some transformations were done from the base datasets:
- We accumulated precipitation over 6 hours instead of the default 1 hour.
- For HRES data, each time step corresponds to the HRES forecast at leadtime 0, essentially providing an "initialisation" from HRES. See HRES-fc0 in the GraphCast paper for further description. Note that a 6h accumulation of precipitation is not available from HRES, so our model taking HRES inputs does not depend on precipitation. However, because our models predict precipitation, we include the ERA5 precipitation in the example data so it can serve as an illustrative example of ground truth.
- We include ERA5 `toa_incident_solar_radiation` in the data. Our model uses the radiation at -6h, 0h and +6h as a forcing term for each 1-step prediction. If the radiation is missing from the data (e.g. in an operational setting), it will be computed using a custom implementation that produces values similar to those in ERA5.

In [ ]:
# @title Get and filter the list of available example datasets

dataset_file_options = [
    name for blob in gcs_bucket.list_blobs(prefix="dataset/")
    if (name := blob.name.removeprefix("dataset/"))]  # Drop empty string.

def data_valid_for_model(
    file_name: str, model_config: graphcast.ModelConfig, task_config: graphcast.TaskConfig):
  file_parts = parse_file_parts(file_name.removesuffix(".nc"))
  return (
      model_config.resolution in (0, float(file_parts["res"])) and
      len(task_config.pressure_levels) == int(file_parts["levels"]) and
      (
          ("total_precipitation_6hr" in task_config.input_variables and
           file_parts["source"] in ("era5", "fake")) or
          ("total_precipitation_6hr" not in task_config.input_variables and
           file_parts["source"] in ("hres", "fake"))
      )
  )

dataset_file = widgets.Dropdown(
    options=[
        (", ".join([f"{k}: {v}" for k, v in parse_file_parts(option.removesuffix(".nc")).items()]), option)
        for option in dataset_file_options
        if data_valid_for_model(option, model_config, task_config)
    ],
    description="Dataset file:",
    layout={"width": "max-content"})
widgets.VBox([
    dataset_file,
    widgets.Label(value="Run the next cell to load the dataset. Rerunning this cell clears your selection and refilters the datasets that match your model.")
])

In [ ]:
# @title Load weather data

if not data_valid_for_model(dataset_file.value, model_config, task_config):
  raise ValueError(
      "Invalid dataset file, rerun the cell above and choose a valid dataset file.")

with gcs_bucket.blob(f"dataset/{dataset_file.value}").open("rb") as f:
  example_batch = xarray.load_dataset(f).compute()

assert example_batch.dims["time"] >= 3  # 2 for input, >=1 for targets

print(", ".join([f"{k}: {v}" for k, v in parse_file_parts(dataset_file.value.removesuffix(".nc")).items()]))

example_batch

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive/')

In [ ]:
%cd /content/drive/MyDrive/Data/

In [ ]:
example_batch_ = xarray.open_dataset('data_graphcast_2020_era5_30_days.nc')

In [ ]:
example_batch_.isel(batch=slice(0,3)).to_netcdf('sample_data.nc')

In [ ]:
example_batch_ = xarray.open_dataset('/content/drive/MyDrive/data_graphcast_2017.nc')

In [ ]:
data_test = xarray.open_dataset('data_graphcast_2020_era5_reforecasts_dates.nc')

In [ ]:
#new way of generating inputs for graphcast model

def createInputDatas(example_batch_,index=0):

    """Input datas for training loop.

  Args:
    Example_batch : xarray.Dataset containing all meteorological variables.

    Index : Index loop indicating which data to extract

  Returns:
    train_inputs, train_targets, train_forcings, train_targets1, train_forcings1

"""
    #extracting data from 7 to 14 days ahead from batch n+1 timestamps slice("168h","336h")
    #extracting data from 14 to 21 days ahead from batch n+1 timestamps slice("336h","504h")
    # from 21 to 28 days slice("504h","672h")
    train_inputs, train_targets, train_forcings = data_utils.extract_inputs_targets_forcings(
        example_batch_.isel(batch=[index+1]), target_lead_times=slice("336h","504h"),
        **dataclasses.asdict(task_config))

    #extracting data from 0 to 7 days ahead from batch n
    #this data is used as input data for FT graphcast model
    train_inputs1, train_targets1, train_forcings1 = data_utils.extract_inputs_targets_forcings(
        example_batch_.isel(batch=[index]), target_lead_times=slice("0h","168h"),
        **dataclasses.asdict(task_config))

    return train_inputs, train_targets, train_forcings, train_targets1, train_forcings1

#def combiningData(train_targets1, train_forcings1):
#
#    return train_targets1.merge(train_forcings1)

def combiningData(first, second):
    return first.merge(second)

def weekly_mean(data,fake_data,input_data=False):
    if input_data:
        return data.mean('time',keepdims=True).assign_coords(fake_data.isel(time=[0]).coords)
    else:
        return data.mean('time',keepdims=True).assign_coords(fake_data.coords)

In [ ]:
# @title Choose data to plot

plot_example_variable = widgets.Dropdown(
    options=example_batch.data_vars.keys(),
    value="2m_temperature",
    description="Variable")
plot_example_level = widgets.Dropdown(
    options=example_batch.coords["level"].values,
    value=500,
    description="Level")
plot_example_robust = widgets.Checkbox(value=True, description="Robust")
plot_example_max_steps = widgets.IntSlider(
    min=1, max=example_batch.dims["time"], value=example_batch.dims["time"],
    description="Max steps")

widgets.VBox([
    plot_example_variable,
    plot_example_level,
    plot_example_robust,
    plot_example_max_steps,
    widgets.Label(value="Run the next cell to plot the data. Rerunning this cell clears your selection.")
])

In [ ]:
# @title Plot example data

plot_size = 7

data = {
    " ": scale(select(example_batch, plot_example_variable.value, plot_example_level.value, plot_example_max_steps.value),
              robust=plot_example_robust.value),
}
fig_title = plot_example_variable.value
if "level" in example_batch[plot_example_variable.value].coords:
  fig_title += f" at {plot_example_level.value} hPa"

plot_data(data, fig_title, plot_size, plot_example_robust.value)


In [ ]:
# @title Choose training and eval data to extract
train_steps = widgets.IntSlider(
    value=1, min=1, max=example_batch.sizes["time"]-2, description="Train steps")
eval_steps = widgets.IntSlider(
    value=example_batch.sizes["time"]-2, min=1, max=example_batch.sizes["time"]-2, description="Eval steps")

widgets.VBox([
    train_steps,
    eval_steps,
    widgets.Label(value="Run the next cell to extract the data. Rerunning this cell clears your selection.")
])

In [ ]:
# @title Extract training and eval data
#defaults parameters for graphcast

train_inputs_fake, train_targets_fake, train_forcings_fake = data_utils.extract_inputs_targets_forcings(
    example_batch, target_lead_times=slice("6h", f"{train_steps.value*6}h"),
    **dataclasses.asdict(task_config))

eval_inputs_fake, eval_targets_fake, eval_forcings_fake = data_utils.extract_inputs_targets_forcings(
    example_batch, target_lead_times=slice("6h", f"{eval_steps.value*6}h"),
    **dataclasses.asdict(task_config))

print("All Examples:  ", example_batch.dims.mapping)
print("Train Inputs:  ", train_inputs_fake.dims.mapping)
print("Train Targets: ", train_targets_fake.dims.mapping)
print("Train Forcings:", train_forcings_fake.dims.mapping)
print("Eval Inputs:   ", eval_inputs_fake.dims.mapping)
print("Eval Targets:  ", eval_targets_fake.dims.mapping)
print("Eval Forcings: ", eval_forcings_fake.dims.mapping)

In [ ]:
# @title Load normalization data

with gcs_bucket.blob("stats/diffs_stddev_by_level.nc").open("rb") as f:
  diffs_stddev_by_level = xarray.load_dataset(f).compute()
with gcs_bucket.blob("stats/mean_by_level.nc").open("rb") as f:
  mean_by_level = xarray.load_dataset(f).compute()
with gcs_bucket.blob("stats/stddev_by_level.nc").open("rb") as f:
  stddev_by_level = xarray.load_dataset(f).compute()

# Run the model

Note that the cell below may take a while (possibly minutes) to run the first time you execute them, because this will include the time it takes for the code to compile. The second time running will be significantly faster.

This use the python loop to iterate over prediction steps, where the 1-step prediction is jitted. This has lower memory requirements than the training steps below, and should enable making prediction with the small GraphCast model on 1 deg resolution data for 4 steps.

# Train the model

The following operations require a large amount of memory and, depending on the accelerator being used, will only fit the very small "random" model on low resolution data. It uses the number of training steps selected above.

The first time executing the cell takes more time, as it include the time to jit the function.

Fine-tuning of Graphcast using optax

In [ ]:
import optax

# modify the gradients function signature
def grads_fn(params, state, inputs, targets, forcings, model_config, task_config):
    def _aux(params, state, i, t, f):
        (loss, diagnostics), next_state = loss_fn.apply(params, state, jax.random.PRNGKey(0), model_config, task_config, i, t, f)
        return loss, (diagnostics, next_state)
    (loss, (diagnostics, next_state)), grads = jax.value_and_grad(_aux, has_aux=True)(params, state, inputs, targets, forcings)
    return loss, diagnostics, next_state, grads

def FineTuning(train_inputs,train_targets,train_forcings,params):

    # remove `with_params` from jitted grads function
    grads_fn_jitted = jax.jit(with_configs(grads_fn))

    # setup optimiser
    lr = 1e-4
    optimiser = optax.adam(lr, b1=0.9, b2=0.999, eps=1e-8)
    opt_state = optimiser.init(params)

    # calculate loss and gradients
    loss, diagnostics, next_state, grads = grads_fn_jitted(params, state, train_inputs, train_targets, train_forcings)

    # update
    updates, opt_state = optimiser.update(grads, opt_state)
    params = optax.apply_updates(params, updates)

    return params,loss

In [ ]:
#fucntions for saving and loading model
import jax.numpy as jnp

def flatten_dict(d, parent_key='', sep='//'):
    items = []
    for k, v in d.items():
        new_key = f"{parent_key}{sep}{k}" if parent_key else k
        if isinstance(v, dict):
            items.extend(flatten_dict(v, new_key, sep=sep).items())
        else:
            items.append((new_key, v))
    return dict(items)

def save_model_params(d, file_path):
    flat_dict = flatten_dict(d)
    # Convert JAX arrays to NumPy for saving
    np_dict = {k: np.array(v) if isinstance(v, jnp.ndarray) else v for k, v in flat_dict.items()}
    np.savez(file_path, **np_dict)

def unflatten_dict(d, sep='//'):
    result_dict = {}
    for flat_key, value in d.items():
        keys = flat_key.split(sep)
        d = result_dict
        for key in keys[:-1]:
            if key not in d:
                d[key] = {}
            d = d[key]
        d[keys[-1]] = value
    return result_dict

def load_model_params(file_path):
    with np.load(file_path, allow_pickle=True) as npz_file:
        # Convert NumPy arrays back to JAX arrays
        jax_dict = {k: jnp.array(v) for k, v in npz_file.items()}
    return unflatten_dict(jax_dict)

#params_path = os.path.join('/content/drive/MyDrive/Data', 'params.npz')

In [ ]:
def train_graphcast(data,params):

    params_path = os.path.join('/content/drive/MyDrive/Data', 'params_4_weeks_ahead_2019.npz')
    epochs = 10
    Loss = []

    for epoch in range(epochs):
        print(f'--------> Epoch n°{epoch}')

        for i in range(data.dims.mapping['batch']-1):
            print(f'batch n°{i+1}')

            train_inputs, train_targets, train_forcings, train_targets1, train_forcings1 = createInputDatas(data,index=i)
            combined_data = combiningData(train_targets1, train_forcings1)

            train_targets_mean_7_days = weekly_mean(train_targets,train_targets_fake)
            train_forcings_mean_7_days = weekly_mean(train_forcings,train_forcings_fake)

            #combining train_inputs so that we have the previous 7 days mean with the data at 00:00:00
            #normally it would be data at -1 day at 18:00:00 and data at 00:00:00
            train_inputs_mean_7_days = weekly_mean(combined_data,train_inputs_fake,True)
            train_inputs_mean_7_days['geopotential_at_surface'] = train_inputs.isel(time=1)['geopotential_at_surface']
            train_inputs_mean_7_days['land_sea_mask'] = train_inputs.isel(time=1)['land_sea_mask']
            train_inputs_mean_7_days = xarray.concat([train_inputs_mean_7_days,train_inputs.isel(time=1)],dim='time')
            train_inputs_mean_7_days['geopotential_at_surface'] = train_inputs_mean_7_days['geopotential_at_surface'].isel(batch=0)
            train_inputs_mean_7_days['land_sea_mask'] = train_inputs_mean_7_days['land_sea_mask'].isel(batch=0)
            train_inputs_mean_7_days['geopotential_at_surface'] = train_inputs_mean_7_days['geopotential_at_surface'].isel(time=0)
            train_inputs_mean_7_days['land_sea_mask'] = train_inputs_mean_7_days['land_sea_mask'].isel(time=0)

            print("Train Inputs:  ", train_inputs_mean_7_days.dims.mapping)
            print("Train Targets: ", train_targets_mean_7_days.dims.mapping)
            print("Train Forcings:", train_forcings_mean_7_days.dims.mapping)

            params, loss = FineTuning(train_inputs_mean_7_days,train_targets_mean_7_days,train_forcings_mean_7_days,params)
            print(f'--------> loss for batch n°{i+1} = {loss}')
            Loss.append(loss)

        print(f'--------> Saving model after epoch n°{epoch}')
        save_model_params(params, params_path)

In [ ]:
#retraining from the start

train_graphcast(example_batch_,params)

In [ ]:
#reloading params from model trained on XX year for retraining on other year

params_path = os.path.join('/content/drive/MyDrive/Data', 'params_4_weeks_ahead_2019.npz')
params = load_model_params(params_path)

train_graphcast(example_batch_,params)

In [ ]:
def GraphcastPredictons(data_test):
      #calling fine-tuned model

      params_path = os.path.join('/content/drive/MyDrive/Data', 'params_3_weeks_ahead_2017_2018_2019.npz')
      params = load_model_params(params_path)

      dates = []
      predictions = []
      for i in range(data_test.dims.mapping['batch']-1):
          print(f'batch n°{i+1}')

          dates.append(data_test.datetime.values[i+1,0])
          train_inputs, train_targets, train_forcings, train_targets1, train_forcings1 = createInputDatas(data_test,index=i)
          combined_data = combiningData(train_targets1, train_forcings1)

          train_targets_mean_7_days = weekly_mean(train_targets,train_targets_fake)
          train_forcings_mean_7_days = weekly_mean(train_forcings,train_forcings_fake)

          #combining train_inputs so that we have the previous 7 days mean with the data at at real time T 00:00:00
          #normally it would be data at -1 day at 18:00:00 and data at 00:00:00
          train_inputs_mean_7_days = weekly_mean(combined_data,train_inputs_fake,True)
          train_inputs_mean_7_days['geopotential_at_surface'] = train_inputs.isel(time=1)['geopotential_at_surface']
          train_inputs_mean_7_days['land_sea_mask'] = train_inputs.isel(time=1)['land_sea_mask']
          train_inputs_mean_7_days = xarray.concat([train_inputs_mean_7_days,train_inputs.isel(time=1)],dim='time')
          train_inputs_mean_7_days['geopotential_at_surface'] = train_inputs_mean_7_days['geopotential_at_surface'].isel(batch=0)
          train_inputs_mean_7_days['land_sea_mask'] = train_inputs_mean_7_days['land_sea_mask'].isel(batch=0)
          train_inputs_mean_7_days['geopotential_at_surface'] = train_inputs_mean_7_days['geopotential_at_surface'].isel(time=0)
          train_inputs_mean_7_days['land_sea_mask'] = train_inputs_mean_7_days['land_sea_mask'].isel(time=0)

          print("Train Inputs:  ", train_inputs_mean_7_days.dims.mapping)
          print("Train Targets: ", train_targets_mean_7_days.dims.mapping)
          print("Train Forcings:", train_forcings_mean_7_days.dims.mapping)

          #removing 'with_params' from jitted functions and inputing manually new parameters
          run_forward_jitted = drop_state(jax.jit(with_configs(run_forward.apply)))
          loss_fn_jitted = drop_state(jax.jit(with_configs(loss_fn.apply)))

          prediction = run_forward_jitted(
                rng=jax.random.PRNGKey(0),
                inputs=train_inputs_mean_7_days,
                targets_template=train_targets_mean_7_days * np.nan,
                forcings=train_forcings_mean_7_days,
                params=params,
                state=state)

          predictions.append(prediction)

          if i in [0,1,2]:
              # @title Loss computation (autoregressive loss over multiple steps)
              loss, diagnostics = loss_fn_jitted(
                  rng=jax.random.PRNGKey(0),
                  inputs=train_inputs_mean_7_days,
                  targets=train_targets_mean_7_days,
                  forcings=train_forcings_mean_7_days,
                  params=params,
                  state=state)
              print("Loss:", float(loss))

      predictions = xarray.concat(predictions,dim='batch')

      return predictions,dates


In [ ]:
predictions,dates = GraphcastPredictons(data_test)

In [ ]:
np.save('/content/drive/MyDrive/Data/predictions_dates_4WeeksAhead_graphcast_2021_reforecasts.npy',dates)

In [ ]:
predictions.to_netcdf('/content/drive/MyDrive/Data/predictions_FT_2019_21_days_2021_reforecasts.nc')